# Query Comparison: SSN Submit Rate & Conversion After Credit

Runs both the **repo query** (`session_level_query`) and the **other query** (pasted below) for the same WoW date range, then compares SSN Submit Rate and Conv After Credit side by side.

Goal: pinpoint exactly where and how much the numbers diverge.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

import pandas as pd
import numpy as np
from datetime import date
from dotenv import load_dotenv
from databricks import sql as databricks_sql
from IPython.display import display, Markdown

load_dotenv(override=True)

_HOST = os.getenv('DATABRICKS_HOST', '')
_TOKEN = os.getenv('DATABRICKS_TOKEN', '')
_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH', '')

def run_query(sql):
    conn = databricks_sql.connect(
        server_hostname=_HOST.replace('https://', '').strip('/'),
        http_path=_HTTP_PATH,
        access_token=_TOKEN,
    )
    cursor = conn.cursor()
    cursor.execute(sql)
    rows = cursor.fetchall()
    cols = [desc[0] for desc in cursor.description]
    conn.close()
    return pd.DataFrame(rows, columns=cols)

def safe_rate(n, d):
    return n / d if d > 0 else 0.0

# WoW comparison dates
CURR_START, CURR_END = date(2026, 3, 16), date(2026, 3, 22)
PRIOR_START, PRIOR_END = date(2026, 3, 9), date(2026, 3, 15)
QUERY_START, QUERY_END = PRIOR_START, CURR_END

DEFAULT_CHANNELS = ['Paid Search', 'Direct', 'Organic', 'pMax']

print(f'Current week: {CURR_START} → {CURR_END}')
print(f'Prior week:   {PRIOR_START} → {PRIOR_END}')
print(f'Query range:  {QUERY_START} → {QUERY_END}')

## 1. Run the REPO query (`session_level_query`)

In [ ]:
query_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'session_level_query'))
with open(query_path) as f:
    repo_sql = f.read()

repo_sql = repo_sql.replace('%sql', '', 1).strip()
repo_sql = repo_sql.replace(
    "('2026-01-01')::date AS start_date",
    f"('{QUERY_START}')::date AS start_date"
)
repo_sql = repo_sql.replace(
    "(CURRENT_DATE)::date AS end_date",
    f"('{QUERY_END}')::date AS end_date"
)

print('Running repo query...')
df_repo = run_query(repo_sql)
print(f'Repo query returned {len(df_repo):,} rows')
print(f'Columns: {list(df_repo.columns)}')

## 2. Run the OTHER query

In [ ]:
# Paste the other query here with date range injected
other_sql = f"""
WITH cte_variables AS (
    SELECT 
          ('{QUERY_START}')::date AS start_date
        , ('{QUERY_END}')::date AS end_date
)

, base_sessions AS (
 SELECT s.web_context_anonymous_id
   , s.webcontext_session_id
   , s.session_start_ts_est
   , s.session_start_date_est
   , s.unique_pages_viewed
   , s.first_page_no_query_string
   , s.first_page_domain
   , s.first_page_referrer_no_query_string
   , s.first_page_referrer
   , s.utility_area
   , s.first_page_referrer_domain
   , s.first_page_url_first_folder
   , s.first_page_url
   , s.isp_name
   , s.is_bot
   , s.device_type
   , s.city
   , s.entered_zip
   , CASE WHEN s.is_product_viewed = 1  THEN 1 ELSE 0 END AS zip_entry
   , s.is_product_viewed
   , s.postal_code
   , s.region
   , s.mover_switcher
   , s.latitude
   , s.longitude
   , case
       when s.site_name = 'www.chooseenergy.com' and (s.Region = 'tx' OR s.first_page_url ILIKE '%texas%') then 'Choose TX'
       when s.site_name = 'www.chooseenergy.com' and (s.Region != 'TX' OR s.region IS NULL AND s.first_page_url NOT ILIKE '%texas%') then 'Choose NTX'
       when s.site_name = 'www.chooseenergy.com' and (s.first_page_url IN ('https://www.chooseenergy.com/', 'https://www.chooseenergy.com/providers/', 'https://www.chooseenergy.com/utilities/') AND s.region = 'tx') then 'Choose TX'
       when s.site_name = 'www.chooseenergy.com' and (s.first_page_url IN ('https://www.chooseenergy.com/', 'https://www.chooseenergy.com/providers/', 'https://www.chooseenergy.com/utilities/') AND s.region != 'tx') then 'Choose NTX'
       when s.site_name = 'saveonenergy.com' then 'SOE'
       when s.site_name = 'choosetexaspower.org' then 'CTXP'
       when s.site_name = 'texaselectricrates.com' then 'TXER'
       else 'Unknown'
       end as website
   , CASE
        WHEN s.campaign ILIKE '%^Display%' THEN 'Display'
        WHEN s.traffic_source = 'Paid' AND s.traffic_channel = 'Search' THEN 'Paid Search'
        WHEN s.traffic_source = 'Paid' AND s.traffic_channel = 'Pmax'   THEN 'pMax'
        WHEN (s.first_page_referrer ILIKE '%chatgpt%' OR s.first_page_url ILIKE '%chatgpt%') or
            s.first_page_referrer ILIKE '%copilot.microsoft%' or
            s.first_page_referrer ILIKE '%perplexity%' or
            s.first_page_referrer ILIKE '%gemini.google%' or 
            s.first_page_referrer ILIKE '%claude%' or
            s.first_page_referrer ILIKE '%andisearch%' or
            s.first_page_referrer ILIKE '%bagoodex%' or
            s.first_page_referrer ILIKE '%brave%' or
            s.first_page_referrer ILIKE '%grok%' THEN 'AI Search'
        WHEN s.traffic_source IN ('Direct')  AND s.traffic_channel IN ('Direct','Search') AND s.transactional_ind = 1 THEN 'Direct'
        WHEN s.traffic_source IN ('Organic') AND s.traffic_channel IN ('Direct','Search') AND s.transactional_ind = 1 THEN 'Organic'
        WHEN s.traffic_source IN ('Organic','Direct') AND s.traffic_channel IN ('Referral') AND s.transactional_ind = 1 THEN 'Referral'
        WHEN s.traffic_source IN ('Affiliate','Affliate') THEN 'Affiliate'
        WHEN s.traffic_source IN ('Email') OR s.traffic_channel IN ('Email') THEN 'Email'
        WHEN s.traffic_source IN ('SMS')   OR s.traffic_channel IN ('SMS')   THEN 'SMS'
        WHEN s.traffic_channel = 'Social' THEN 'Social'
        WHEN s.traffic_channel = 'Internal' OR s.traffic_source = 'Internal' THEN 'Internal'
        ELSE 'Other'
    END AS marketing_channel,
    CASE WHEN first_page_no_query_string IN ('www.chooseenergy.com', 'www.saveonenergy.com', 'www.choosetexaspower.org', 'www.texaselectricrates.com') THEN 1 ELSE 0 END as is_lp,
    CASE WHEN s.landing_page ILIKE '%Provider%' THEN 'Provider'
         WHEN s.landing_page ILIKE '%Resources%' THEN 'Resources'
         WHEN s.landing_page ILIKE '%Grid%'      THEN 'Grid'
         ELSE s.landing_page END as landing_page_type  
 FROM energy_prod.energy.v_sessions s
 CROSS JOIN cte_variables var
 WHERE s.session_start_date_est between var.start_date and var.end_date
   AND s.bot_ip_ind = 0
   AND s.is_bot = false
   AND (s.is_internal_ip = FALSE OR s.is_internal_ip IS NULL)
   AND COALESCE(s.company_id, 25) = 25
   AND s.first_page_url NOT ILIKE '%solar-energy%'
   AND s.first_page_url NOT ILIKE '%resources%'
   AND (s.non_texas_ind = 0 or s.campaign ilike '%a:SOE%')
   AND (s.category IS NULL OR s.category != 'Non-Texas')
   AND (LOWER(s.region) = 'tx' OR s.first_page_url ILIKE '%texas%' OR s.site_name NOT ILIKE '%chooseenergy%')
)
, cart_orders as (
  select o.order_id,  
    max(rev.gcv_new_v2) as order_gcv, 
    max(o.partner_name) order_partner_name, 
    max(o.product_name) order_product_name,
    max(o.term) order_term_length, 
    max(o.order_date_est) order_date_est, 
    1 as cart_order
  from energy_prod.energy.v_orders o
  left join energy_prod.energy.v_orders_gcv rev ON o.order_id = rev.order_id
  cross join cte_variables var
  where o.order_date_est >= var.start_date
    and o.order_type = 'Cart'
    and o.anonymous_id is not null
    AND o.order_date_est BETWEEN var.start_date AND var.end_date
    AND o.txorder_ind = 1
    AND COALESCE(o.company_id,25)=25
    AND (o.tenant_id != 'src_1ax2zVVJJ2cB9aBFEnVcRPog5Pe' OR o.tenant_id IS NULL)
    AND o.partner_name <> 'TXU Branded Energy'
  group by all
)
, cartsession_id_base AS (
 SELECT c.cartsession_id
   , c.websession_id
   , co.order_gcv 
   , c.first_partner_name
   , c.first_product_name
   , c.last_partner_name
   , c.last_product_name
   , co.order_partner_name
   , co.order_product_name
   , co.order_term_length
   , co.cart_order
 FROM energy_prod.energy.v_carts c
 LEFT JOIN cart_orders co ON c.cartsession_id = co.order_id
 CROSS JOIN cte_variables var
 WHERE c.cartsession_start_date_est BETWEEN var.start_date and var.end_date
  AND c.txmarketplace_ind = 1
  AND c.bot_ip_ind = 0
  AND c.is_internal_ip = FALSE
  AND COALESCE(c.company_id, 25) = 25
)
, pivot_clicked AS (
 SELECT fc.correlationId cart_instance_id, 1 pivot_clicked
 FROM lakehouse_production.energy.event_redventures_usertracking_formcontinued fc
 CROSS JOIN cte_variables var
 WHERE fc._date >= var.start_date
   AND fc.step_context_step_name IN ('standard-pivot', 'prepaid-pivot','no-deposit-plans')
 GROUP BY fc.correlationId
 UNION
 SELECT pc.cartContext.cartInstanceId cart_instance_id, 1 pivot_clicked
 FROM lakehouse_production.energy.event_redventures_cart_productclicked pc
 CROSS JOIN cte_variables var
 WHERE pc._date >= var.start_date
   AND (pc.product_location ILIKE '%pivot%' OR pc.product_location ILIKE 'no-deposit-plans')
 GROUP BY pc.cartContext.cartInstanceId
)
, cart_instance_join AS (
 SELECT cs.cart_instance_id, MAX(cs.cart_session_id) cart_session_id
 FROM lakehouse_production.energy.event_redventures_cart_cartstarted cs
 CROSS JOIN cte_variables var
 WHERE cs._date >= var.start_date
 GROUP BY cs.cart_instance_id
)
, pivot_click_final AS (
 SELECT cij.cart_session_id, MAX(pvc.pivot_clicked) pivot_clicked
 FROM cart_instance_join cij
 INNER JOIN pivot_clicked pvc ON pvc.cart_instance_id = cij.cart_instance_id
 GROUP BY cij.cart_session_id
)
, cart_first_credit_run_calc AS (
 SELECT
   cr.client_session_id,
   cr._timeStamp AS sent_at,
   cr.creditScore,
   cr.creditRunId,
   cr.providerName,
   CASE WHEN cr.passSelectedThreshold = true THEN 'pass'
     WHEN cr.passSelectedThreshold = false THEN 'fail'
     ELSE 'other' END AS provider_pass,
   ROW_NUMBER() OVER (PARTITION BY cr.client_session_id ORDER BY cr._timeStamp ASC) AS rn
 FROM lakehouse_production.energy.event_energy_creditresult cr
 CROSS JOIN cte_variables var
 WHERE cr._date >= var.start_date
   AND cr.client_session_id NOT LIKE 'WR%'
   AND cr.web_session_id IS NOT NULL
)
, cart_first_credit_run AS (
 SELECT client_session_id,
   creditScore first_run_credit_score,
   providerName first_run_provider_name,
   provider_pass first_run_provider_pass
 FROM cart_first_credit_run_calc
 WHERE rn = 1
)
, cart_agged_credit_run AS (
 SELECT cr.client_session_id,
   MIN(cr._timeStamp) sent_at,
   collect_list(DISTINCT cr.creditScore) unique_scores,
   MAX(cr.creditScore) max_score,
   COUNT(DISTINCT cr.creditRunId) credit_runs,
   MAX(size(array_distinct(cr.providerFailed))) providers_failed,
   MAX(size(array_distinct(cr.providerPassed))) providers_passed,
   MAX(CASE WHEN cr.passLowestThreshold = true THEN 1 ELSE 0 END) ucc_pass,
   MAX(CASE WHEN cr.passSelectedThreshold = true THEN 1 ELSE 0 END) provider_pass
 FROM lakehouse_production.energy.event_energy_creditresult cr
 CROSS JOIN cte_variables var
 WHERE cr._date >= var.start_date
   AND cr.client_session_id NOT LIKE 'WR%'
   AND cr.web_session_id IS NOT NULL
 GROUP BY cr.client_session_id
)
, cart_final_credit_run AS (
 SELECT fcr.client_session_id
   , first_run_credit_score
   , first_run_provider_name
   , first_run_provider_pass
   , CASE WHEN fcr.first_run_provider_pass = 'fail' THEN 1 ELSE 0 END credit_fail
   , acr.unique_scores AS cart_unique_credit_scores
   , acr.max_score AS cart_max_credit_score
   , acr.credit_runs AS cart_credit_runs
   , acr.providers_failed AS cart_providers_failed_credit
   , acr.providers_passed AS cart_providers_passed_credit
   , acr.ucc_pass AS cart_ucc_pass_credit
 FROM cart_first_credit_run fcr
 LEFT JOIN cart_agged_credit_run acr ON fcr.client_session_id = acr.client_session_id
)
, pivot_triggers_two_alt AS (
 SELECT integration_context_client_session_id
 FROM lakehouse_production.energy.event_energy_pivotresult
 CROSS JOIN cte_variables var
 WHERE _date > var.start_date
 GROUP BY integration_context_client_session_id
)
, rogs_indicator as (
  SELECT session_id,
    MAX(CASE WHEN webElement.elementType = 'Cart CTA' AND webElement.location = 'ROG' AND webElement.text = 'Check Availability' THEN 1 ELSE 0 END) as rogs_ind
  FROM lakehouse_production.energy.event_redventures_usertracking_elementclicked
  CROSS JOIN cte_variables var
  WHERE _date > var.start_date
  GROUP BY session_id
)
, step_completions AS (
  SELECT cartsession_id_completions,
    MAX(page_1_completion) AS page_1_completion,
    MAX(customer_info_completion) AS customer_info_completion,
    MAX(ssn_completion) AS ssn_completion
  FROM (
    SELECT fc2.order_session_id AS cartsession_id_completions,
      MAX(CASE WHEN fc2.cartContext_pageReferrer ILIKE '%mover-switcher%' THEN 1 END) AS page_1_completion,
      MAX(CASE WHEN fc2.formContext.formName = 'PersonalInfo' THEN 1 END) AS customer_info_completion,
      MAX(CASE WHEN fc2.formContext.formName = 'CreditVerification' THEN 1 END) AS ssn_completion
    FROM lakehouse_production.energy.event_redventures_cart_formcontinued fc2
    CROSS JOIN cte_variables var
    WHERE fc2._date > var.start_date
    GROUP BY fc2.order_session_id
    UNION ALL
    SELECT cs.cart_session_id AS cartsession_id_completions,
      MAX(CASE WHEN fc3.step_context_step_name = 'connection-info' THEN 1 END) AS page_1_completion,
      MAX(CASE WHEN fc3.step_context_step_name = 'contact-info' THEN 1 END) AS customer_info_completion,
      MAX(CASE WHEN fc3.step_context_step_name = 'credit-verification' THEN 1 END) AS ssn_completion
    FROM lakehouse_production.energy.event_redventures_usertracking_formcontinued fc3
    JOIN (
      SELECT cart_instance_id, MAX(cart_session_id) AS cart_session_id
      FROM lakehouse_production.energy.event_redventures_cart_cartstarted
      GROUP BY cart_instance_id
    ) cs ON cs.cart_instance_id = fc3.correlationId
    CROSS JOIN cte_variables var
    WHERE fc3.form_context_form_name = 'Cart' AND fc3._date > var.start_date
    GROUP BY cs.cart_session_id
  ) z
  GROUP BY cartsession_id_completions
)
, attempts AS (
  SELECT qr.client_session_id, qr.web_session_id, qr.response, qr.providerName, qr.step, qr._timeStamp,
         ROW_NUMBER() OVER (PARTITION BY qr.client_session_id, qr.step ORDER BY qr._timeStamp) rn
  FROM lakehouse_production.energy.event_energy_qualificationresult qr
  CROSS JOIN cte_variables var
  WHERE qr._date >= var.start_date
)
, first_midflow AS (
  SELECT client_session_id, web_session_id, response AS midflow_response, providerName AS midflow_provider
  FROM attempts WHERE step = 'MID-FLOW' AND rn = 1
)
, first_enrollment AS (
  SELECT client_session_id, web_session_id, response AS enrollment_response, providerName AS enrollment_provider
  FROM attempts WHERE step = 'ENROLLMENT' AND rn = 1
)
, agg_qual AS (
  SELECT qr.client_session_id, qr.web_session_id,
         SUM(CASE WHEN qr.response = 'FAILURE' THEN 1 ELSE 0 END) AS qual_failures,
         SUM(CASE WHEN qr.response = 'SUCCESS' THEN 1 ELSE 0 END) AS qual_successes
  FROM lakehouse_production.energy.event_energy_qualificationresult qr
  CROSS JOIN cte_variables var
  WHERE qr._date >= var.start_date
  GROUP BY qr.client_session_id, qr.web_session_id
)
, qual_final AS (
  SELECT COALESCE(fm.client_session_id, fe.client_session_id, aq.client_session_id) AS client_session_id,
    COALESCE(fm.web_session_id, fe.web_session_id, aq.web_session_id) AS web_session_id,
    fm.midflow_response, fe.enrollment_response,
    CASE WHEN fm.midflow_response = 'FAILURE' OR fe.enrollment_response = 'FAILURE' THEN 1 ELSE 0 END AS qual_fail,
    aq.qual_failures, aq.qual_successes
  FROM first_midflow fm
  FULL JOIN first_enrollment fe ON fm.client_session_id = fe.client_session_id AND fm.web_session_id = fe.web_session_id
  LEFT JOIN agg_qual aq ON COALESCE(fm.client_session_id, fe.client_session_id) = aq.client_session_id
    AND COALESCE(fm.web_session_id, fe.web_session_id) = aq.web_session_id
)
, first_fail AS (
  SELECT vf.sessionId, vf.voltStepFailed AS first_fail_reason,
         ROW_NUMBER() OVER (PARTITION BY vf.sessionId ORDER BY vf._timeStamp) rn
  FROM lakehouse_production.energy.event_energy_voltfailure vf
  CROSS JOIN cte_variables var WHERE vf._date >= var.start_date
)
, agg_fail AS (
  SELECT vf.sessionId,
         SUM(CASE WHEN vf.voltStepFailed = 'Negative List' THEN 1 ELSE 0 END) AS negative_list_fail_count,
         SUM(CASE WHEN vf.voltStepFailed = 'SEON' THEN 1 ELSE 0 END) AS seon_fail_count
  FROM lakehouse_production.energy.event_energy_voltfailure vf
  CROSS JOIN cte_variables var WHERE vf._date >= var.start_date
  GROUP BY vf.sessionId
)
, volt_fail AS (
  SELECT ff.sessionId AS client_session_id, ff.first_fail_reason,
         af.negative_list_fail_count, af.seon_fail_count, 1 AS volt_fail
  FROM first_fail ff JOIN agg_fail af ON af.sessionId = ff.sessionId WHERE ff.rn = 1
)
, review_page AS (
  SELECT cartsession_id_review, MAX(clicked_review_page_cta) AS clicked_review_page_cta
  FROM (
    SELECT cs.cart_session_id AS cartsession_id_review,
           MAX(CASE WHEN fc.cartcontext.pagereferrer ILIKE '%review%' THEN 1 END) AS clicked_review_page_cta
    FROM lakehouse_production.energy.event_redventures_cart_formsubmitted fc
    JOIN (SELECT cart_instance_id, MAX(cart_session_id) AS cart_session_id FROM lakehouse_production.energy.event_redventures_cart_cartstarted GROUP BY cart_instance_id) cs ON fc.cart_instance_id = cs.cart_instance_id
    CROSS JOIN cte_variables var WHERE fc._date > var.start_date
    GROUP BY cs.cart_session_id
    UNION ALL
    SELECT cs.cart_session_id AS cartsession_id_review,
           MAX(CASE WHEN fc3.step_context_step_name = 'review' THEN 1 END) AS clicked_review_page_cta
    FROM lakehouse_production.energy.event_redventures_usertracking_formcontinued fc3
    JOIN (SELECT cart_instance_id, MAX(cart_session_id) AS cart_session_id FROM lakehouse_production.energy.event_redventures_cart_cartstarted GROUP BY cart_instance_id) cs ON cs.cart_instance_id = fc3.correlationId
    CROSS JOIN cte_variables var WHERE fc3._date > var.start_date
    GROUP BY cs.cart_session_id
  ) z
  GROUP BY cartsession_id_review
)
, cart_session_agg AS (
  SELECT s.webcontext_session_id,
    COUNT(DISTINCT c.cartsession_id) AS cart_entries,
    MAX(CASE WHEN c.cartsession_id IS NOT NULL THEN 1 ELSE 0 END) AS has_cart,
    MAX(c.first_partner_name) AS first_partner_name,
    MAX(c.first_product_name) AS first_product_name,
    MAX(sc.page_1_completion) AS cart_page1_done,
    MAX(sc.customer_info_completion) AS cart_customer_info_done,
    MAX(sc.ssn_completion) AS cart_ssn_done,
    MAX(fcr.cart_credit_runs) AS cart_credit_runs,
    MAX(fcr.cart_max_credit_score) AS cart_max_credit_score,
    MAX(fcr.cart_ucc_pass_credit) AS cart_ucc_pass,
    MAX(fcr.cart_providers_passed_credit) AS cart_providers_passed,
    MAX(fcr.cart_providers_failed_credit) AS cart_providers_failed,
    MAX(CASE WHEN fcr.first_run_provider_pass = 'pass' THEN 1 ELSE 0 END) AS cart_provider_pass,
    MAX(qf.qual_fail) AS cart_qual_fail,
    MAX(v.volt_fail) AS cart_volt_fail,
    SUM(c.cart_order) AS cart_order,
    COUNT(DISTINCT CASE WHEN c.cart_order = 1 THEN c.cartsession_id END) AS cart_order2,
    SUM(c.order_gcv) AS cart_gcv
  FROM base_sessions s
  LEFT JOIN cartsession_id_base c ON c.websession_id = s.webcontext_session_id
  LEFT JOIN pivot_click_final pcf ON pcf.cart_session_id = c.cartsession_id
  LEFT JOIN cart_final_credit_run fcr ON fcr.client_session_id = c.cartsession_id
  LEFT JOIN rogs_indicator ri ON ri.session_id = s.webcontext_session_id
  LEFT JOIN step_completions sc ON sc.cartsession_id_completions = c.cartsession_id
  LEFT JOIN qual_final qf ON qf.client_session_id = c.cartsession_id
  LEFT JOIN volt_fail v ON v.client_session_id = c.cartsession_id
  LEFT JOIN pivot_triggers_two_alt ptt ON ptt.integration_context_client_session_id = c.cartsession_id
  LEFT JOIN review_page rev ON rev.cartsession_id_review = c.cartsession_id
  GROUP BY s.webcontext_session_id
)
, phone_session_agg AS (
  SELECT s.webcontext_session_id,
    COUNT(DISTINCT p.call_id) AS call_count,
    MAX(CASE WHEN p.call_id IS NOT NULL THEN 1 ELSE 0 END) AS has_call,
    COUNT(DISTINCT CASE WHEN p.ib_gross_ind > 0 THEN p.call_id END) as gross_calls,
    COUNT(DISTINCT CASE WHEN p.ib_queue_ind > 0 THEN p.call_id END) AS queue_calls,
    SUM(co.phone_order) AS phone_order
  FROM base_sessions s
  LEFT JOIN (
    SELECT c.call_id, c.web_session_id, c.mover_switcher call_mover_switcher,
      c.talk_time_minutes, c.ib_gross_ind, c.ib_queue_ind, c.abandon_ind, c.ibs_net_ind
    FROM energy_prod.energy.v_calls c
    CROSS JOIN cte_variables var
    WHERE c.call_date_est BETWEEN var.start_date AND var.end_date AND c.web_session_id IS NOT NULL
  ) p ON p.web_session_id = s.webcontext_session_id
  LEFT JOIN (
    SELECT o.call_id, count(distinct o.order_id) AS phone_order
    FROM energy_prod.energy.v_orders o
    CROSS JOIN cte_variables var
    WHERE o.order_date_est BETWEEN var.start_date AND var.end_date AND o.order_type = 'Phone'
    GROUP BY o.call_id
  ) co ON co.call_id = p.call_id
  GROUP BY s.webcontext_session_id
)
, final AS (
  SELECT
    1 as session,
    bs.webcontext_session_id,
    bs.session_start_date_est,
    bs.website,
    bs.device_type,
    bs.marketing_channel,
    bs.mover_switcher,
    bs.zip_entry,
    c.has_cart,
    c.cart_ssn_done,
    c.cart_order,
    c.cart_order2,
    CASE WHEN p.gross_calls >= 1 THEN 1 ELSE 0 END as gross_call,
    CASE WHEN p.queue_calls >= 1 THEN 1 ELSE 0 END AS queue_call,
    CASE WHEN p.phone_order >= 1 THEN 1 ELSE 0 END AS phone_order
  FROM base_sessions bs
  LEFT JOIN cart_session_agg c ON bs.webcontext_session_id = c.webcontext_session_id
  LEFT JOIN phone_session_agg p ON bs.webcontext_session_id = p.webcontext_session_id
)
SELECT * FROM final
"""

print('Running other query...')
df_other = run_query(other_sql)
print(f'Other query returned {len(df_other):,} rows')

## 3. Normalize both DataFrames the same way

In [ ]:
def normalize(df, label):
    """Coerce types, but do NOT clip — keep raw values for comparison."""
    df = df.copy()
    if 'session_start_date_est' in df.columns:
        df['session_start_date_est'] = pd.to_datetime(df['session_start_date_est']).dt.date
    for col in ['session', 'has_cart', 'cart_ssn_done', 'cart_order', 'cart_order2',
                'phone_order', 'zip_entry', 'gross_call', 'queue_call']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    if 'marketing_channel' in df.columns:
        df = df[df['marketing_channel'].isin(DEFAULT_CHANNELS)]
    print(f'{label}: {len(df):,} rows after channel filter')
    print(f'  cart_order value_counts:\n{df["cart_order"].value_counts().sort_index().to_string()}')
    print(f'  cart_order > 1: {(df["cart_order"] > 1).sum():,} sessions')
    print(f'  cart_ssn_done value_counts:\n{df["cart_ssn_done"].value_counts().sort_index().to_string()}')
    print(f'  has_cart value_counts:\n{df["has_cart"].value_counts().sort_index().to_string()}')
    print()
    return df

df_repo_n = normalize(df_repo, 'REPO')
df_other_n = normalize(df_other, 'OTHER')

## 4. Compare KPIs — RAW (no clipping)

In [ ]:
def compute_kpis(df, label, curr_start, curr_end, prior_start, prior_end):
    curr = df[(df['session_start_date_est'] >= curr_start) & (df['session_start_date_est'] <= curr_end)]
    prior = df[(df['session_start_date_est'] >= prior_start) & (df['session_start_date_est'] <= prior_end)]
    
    results = {}
    kpis = {
        'SSN Submit Rate': ('cart_ssn_done', 'has_cart'),
        'Conv After Credit (raw cart_order)': ('cart_order', 'cart_ssn_done'),
    }
    if 'cart_order2' in df.columns:
        kpis['Conv After Credit (cart_order2)'] = ('cart_order2', 'cart_ssn_done')
    
    for kpi_name, (num_col, den_col) in kpis.items():
        if num_col not in df.columns or den_col not in df.columns:
            continue
        c_num, c_den = curr[num_col].sum(), curr[den_col].sum()
        p_num, p_den = prior[num_col].sum(), prior[den_col].sum()
        c_rate = safe_rate(c_num, c_den)
        p_rate = safe_rate(p_num, p_den)
        results[kpi_name] = {
            'curr_num': int(c_num), 'curr_den': int(c_den), 'curr_rate': c_rate,
            'prior_num': int(p_num), 'prior_den': int(p_den), 'prior_rate': p_rate,
            'delta_pp': (c_rate - p_rate) * 100,
        }
    return results

repo_kpis = compute_kpis(df_repo_n, 'REPO', CURR_START, CURR_END, PRIOR_START, PRIOR_END)
other_kpis = compute_kpis(df_other_n, 'OTHER', CURR_START, CURR_END, PRIOR_START, PRIOR_END)

# Build comparison table
rows = []
all_kpi_names = sorted(set(list(repo_kpis.keys()) + list(other_kpis.keys())))
for kpi in all_kpi_names:
    r = repo_kpis.get(kpi, {})
    o = other_kpis.get(kpi, {})
    rows.append({
        'KPI': kpi,
        'REPO Curr Num': r.get('curr_num', ''),
        'REPO Curr Den': r.get('curr_den', ''),
        'REPO Curr Rate': f"{r.get('curr_rate', 0)*100:.4f}%" if r else '',
        'REPO Prior Rate': f"{r.get('prior_rate', 0)*100:.4f}%" if r else '',
        'REPO Delta (pp)': f"{r.get('delta_pp', 0):+.4f}" if r else '',
        'OTHER Curr Num': o.get('curr_num', ''),
        'OTHER Curr Den': o.get('curr_den', ''),
        'OTHER Curr Rate': f"{o.get('curr_rate', 0)*100:.4f}%" if o else '',
        'OTHER Prior Rate': f"{o.get('prior_rate', 0)*100:.4f}%" if o else '',
        'OTHER Delta (pp)': f"{o.get('delta_pp', 0):+.4f}" if o else '',
    })

comparison = pd.DataFrame(rows)
display(Markdown('### Raw comparison (no clipping on either side)'))
display(comparison.style.hide(axis='index'))

## 5. Compare with cart_order clipped to 0/1 (how Streamlit app does it)

In [ ]:
def compute_kpis_clipped(df, curr_start, curr_end, prior_start, prior_end):
    df = df.copy()
    df['cart_order_clipped'] = df['cart_order'].clip(upper=1)
    df['phone_order_clipped'] = df['phone_order'].clip(upper=1) if 'phone_order' in df.columns else 0
    
    curr = df[(df['session_start_date_est'] >= curr_start) & (df['session_start_date_est'] <= curr_end)]
    prior = df[(df['session_start_date_est'] >= prior_start) & (df['session_start_date_est'] <= prior_end)]
    
    results = {}
    kpis = {
        'SSN Submit Rate': ('cart_ssn_done', 'has_cart'),
        'Conv After Credit (clipped)': ('cart_order_clipped', 'cart_ssn_done'),
        'Conv After Credit (raw)': ('cart_order', 'cart_ssn_done'),
    }
    for kpi_name, (num_col, den_col) in kpis.items():
        c_num, c_den = curr[num_col].sum(), curr[den_col].sum()
        p_num, p_den = prior[num_col].sum(), prior[den_col].sum()
        c_rate = safe_rate(c_num, c_den)
        p_rate = safe_rate(p_num, p_den)
        results[kpi_name] = {
            'curr_rate': c_rate, 'prior_rate': p_rate,
            'delta_pp': (c_rate - p_rate) * 100,
        }
    return results

repo_clipped = compute_kpis_clipped(df_repo_n, CURR_START, CURR_END, PRIOR_START, PRIOR_END)
other_clipped = compute_kpis_clipped(df_other_n, CURR_START, CURR_END, PRIOR_START, PRIOR_END)

rows2 = []
for kpi in repo_clipped:
    r = repo_clipped[kpi]
    o = other_clipped.get(kpi, {})
    rows2.append({
        'KPI': kpi,
        'REPO Curr': f"{r['curr_rate']*100:.4f}%",
        'REPO Prior': f"{r['prior_rate']*100:.4f}%",
        'REPO Delta (pp)': f"{r['delta_pp']:+.4f}",
        'OTHER Curr': f"{o.get('curr_rate', 0)*100:.4f}%" if o else '',
        'OTHER Prior': f"{o.get('prior_rate', 0)*100:.4f}%" if o else '',
        'OTHER Delta (pp)': f"{o.get('delta_pp', 0):+.4f}" if o else '',
    })

display(Markdown('### Clipped vs Raw comparison'))
display(pd.DataFrame(rows2).style.hide(axis='index'))

## 6. Session-level join: find sessions that differ between queries

In [ ]:
# Check if there's a session count difference
repo_ids = set(df_repo_n['webcontext_session_id'].unique())
other_ids = set(df_other_n['webcontext_session_id'].unique())

print(f'REPO unique sessions:  {len(repo_ids):,}')
print(f'OTHER unique sessions: {len(other_ids):,}')
print(f'In REPO only:          {len(repo_ids - other_ids):,}')
print(f'In OTHER only:         {len(other_ids - repo_ids):,}')
print(f'In both:               {len(repo_ids & other_ids):,}')

In [ ]:
# For sessions in both, compare cart_order, cart_ssn_done, has_cart values
common_ids = repo_ids & other_ids

repo_common = df_repo_n[df_repo_n['webcontext_session_id'].isin(common_ids)][[
    'webcontext_session_id', 'has_cart', 'cart_ssn_done', 'cart_order'
]].set_index('webcontext_session_id').add_prefix('repo_')

other_common = df_other_n[df_other_n['webcontext_session_id'].isin(common_ids)][[
    'webcontext_session_id', 'has_cart', 'cart_ssn_done', 'cart_order'
]].set_index('webcontext_session_id').add_prefix('other_')

joined = repo_common.join(other_common, how='inner')

# Find mismatches
has_cart_diff = joined[joined['repo_has_cart'] != joined['other_has_cart']]
ssn_diff = joined[joined['repo_cart_ssn_done'] != joined['other_cart_ssn_done']]
order_diff = joined[joined['repo_cart_order'] != joined['other_cart_order']]

print(f'Sessions with has_cart mismatch:    {len(has_cart_diff):,}')
print(f'Sessions with cart_ssn_done mismatch: {len(ssn_diff):,}')
print(f'Sessions with cart_order mismatch:    {len(order_diff):,}')

if len(order_diff) > 0:
    print(f'\nSample cart_order mismatches:')
    display(order_diff.head(10))

if len(ssn_diff) > 0:
    print(f'\nSample cart_ssn_done mismatches:')
    display(ssn_diff.head(10))

if len(has_cart_diff) > 0:
    print(f'\nSample has_cart mismatches:')
    display(has_cart_diff.head(10))

## 7. Summary

In [ ]:
display(Markdown(f"""
### Findings

| Check | Result |
|-------|--------|
| Session count — REPO | {len(repo_ids):,} |
| Session count — OTHER | {len(other_ids):,} |
| Sessions in REPO only | {len(repo_ids - other_ids):,} |
| Sessions in OTHER only | {len(other_ids - repo_ids):,} |
| `has_cart` mismatches | {len(has_cart_diff):,} |
| `cart_ssn_done` mismatches | {len(ssn_diff):,} |
| `cart_order` mismatches | {len(order_diff):,} |

If the session counts differ, that means the `base_sessions` filters or `initiative_master` join
in the repo query is adding/removing sessions. If the column values differ on common sessions,
that points to a join or aggregation difference in `cart_session_agg`.
"""))